In [ ]:
import sys; sys.path.insert(0, "..")
import matplotlib
matplotlib.use("Agg")
import pyulog
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from pathlib import Path
from pid_optimizer.plant_model import PlantModel, Inertia, Drag

LOG_PATH = "../px4_logs/Hermit/testes_PID_position_31-08/log_12_2024-8-30-16-21-54.ulg"
ARTIFACT_PATH = "../artifacts/plant_model.json"
Path("../artifacts").mkdir(exist_ok=True)

log = pyulog.ULog(LOG_PATH, disable_str_exceptions=True)
t0 = log.start_timestamp

def to_df(log, topic, multi_id=0):
    d = next(d for d in log.data_list if d.name == topic and d.multi_id == multi_id)
    df = pd.DataFrame({f: d.data[f] for f in d.data})
    df["t_s"] = (df["timestamp"] - t0) / 1e6
    return df.sort_values("t_s").reset_index(drop=True)

In [ ]:
vel = to_df(log, "vehicle_local_position")
act = to_df(log, "actuator_outputs")

t_ref = act["t_s"].values
vz_interp = np.interp(t_ref, vel["t_s"].values, vel["vz"].values)

hover_mask = np.abs(vz_interp) < 0.15
print(f"Hover fraction: {hover_mask.mean():.1%}")

hover_act = act[hover_mask]
pwm_cols = [c for c in act.columns if c.startswith("output[")][:4]
cmds = hover_act[pwm_cols].values
cmd_min, cmd_max = 1000.0, 2000.0
cmds_norm = np.clip((cmds - cmd_min) / (cmd_max - cmd_min), 0.0, 1.0)

MASS_KG = 1.8
G = 9.81
sum_sq = np.mean(np.sum(cmds_norm ** 2, axis=1))
kT = MASS_KG * G / sum_sq
print(f"Estimated kT = {kT:.6f} N per (normalized_cmd^2) per motor")

In [ ]:
rates_df = to_df(log, "vehicle_angular_velocity")
t_rate = rates_df["t_s"].values
p_rate = np.interp(t_ref, t_rate, rates_df["xyz[0]"].values)
q_rate = np.interp(t_ref, t_rate, rates_df["xyz[1]"].values)
r_rate = np.interp(t_ref, t_rate, rates_df["xyz[2]"].values)

alpha_roll  = np.gradient(p_rate, t_ref)
alpha_pitch = np.gradient(q_rate, t_ref)
alpha_yaw   = np.gradient(r_rate, t_ref)

cmds_all = act[pwm_cols].values
cmds_all_norm = np.clip((cmds_all - cmd_min) / (cmd_max - cmd_min), 0.0, 1.0)
arm = 0.17

tau_roll  = kT * arm * (-cmds_all_norm[:,0] + cmds_all_norm[:,1]
                        + cmds_all_norm[:,2] - cmds_all_norm[:,3])
tau_pitch = kT * arm * (-cmds_all_norm[:,0] - cmds_all_norm[:,1]
                        + cmds_all_norm[:,2] + cmds_all_norm[:,3])
tau_yaw   = 0.1 * kT * arm * (-cmds_all_norm[:,0] + cmds_all_norm[:,1]
                               - cmds_all_norm[:,2] + cmds_all_norm[:,3])

mask_roll  = np.abs(alpha_roll)  > 0.5
mask_pitch = np.abs(alpha_pitch) > 0.5
mask_yaw   = np.abs(alpha_yaw)   > 0.1
Ixx = np.median(tau_roll[mask_roll]  / alpha_roll[mask_roll])   if mask_roll.any()  else 0.012
Iyy = np.median(tau_pitch[mask_pitch]/ alpha_pitch[mask_pitch]) if mask_pitch.any() else 0.013
Izz = np.median(tau_yaw[mask_yaw]   / alpha_yaw[mask_yaw])     if mask_yaw.any()   else 0.022
print(f"Ixx={Ixx:.5f}  Iyy={Iyy:.5f}  Izz={Izz:.5f}  kg·m²")
# Sanity clamp: inertia must be positive and physically plausible for a 1.8kg quad
# Noisy yaw torque estimation often produces wrong-sign Izz; fall back to defaults
Ixx = float(np.clip(abs(Ixx), 0.005, 0.10))
Iyy = float(np.clip(abs(Iyy), 0.005, 0.10))
Izz = float(np.clip(abs(Izz), 0.010, 0.10))
# Enforce Izz >= Ixx (flat quad: z-axis inertia dominates)
Izz = max(Izz, Ixx + Iyy - 0.005)
print(f"After clamping: Ixx={Ixx:.5f}  Iyy={Iyy:.5f}  Izz={Izz:.5f}  kg·m²")

In [ ]:
m0 = act[pwm_cols[0]].values
m0_norm = np.clip((m0 - cmd_min) / (cmd_max - cmd_min), 0.0, 1.0)
steps = np.where(np.diff(m0_norm) > 0.1)[0]

tau_motor = 0.03
if len(steps) > 0:
    i0 = steps[0]
    seg_t = t_ref[i0:i0+100] - t_ref[i0]
    seg_y = m0_norm[i0:i0+100]
    y_final = seg_y[-1] if seg_y[-1] > 0 else 1.0
    def first_order(t, tau): return y_final * (1 - np.exp(-t / tau))
    try:
        popt, _ = curve_fit(first_order, seg_t, seg_y, p0=[0.03], bounds=(0.005, 0.2))
        tau_motor = float(popt[0])
    except Exception:
        pass
print(f"Motor time constant τ = {tau_motor*1000:.1f} ms")

In [ ]:
thrust_total = kT * np.sum(cmds_all_norm**2, axis=1)
glide_mask = thrust_total < (0.1 * MASS_KG * G)
vx_interp = np.interp(t_ref, vel["t_s"].values, vel["vx"].values)
vy_interp = np.interp(t_ref, vel["t_s"].values, vel["vy"].values)

kD_xy, kD_z = 0.15, 0.20
if glide_mask.sum() > 50:
    v_xy = np.sqrt(vx_interp[glide_mask]**2 + vy_interp[glide_mask]**2)
    a_xy = np.gradient(v_xy, t_ref[glide_mask])
    valid = v_xy > 0.1
    if valid.sum() > 20:
        kD_xy = float(np.median(-MASS_KG * a_xy[valid] / v_xy[valid]))
        kD_xy = np.clip(kD_xy, 0.01, 2.0)
print(f"kD_xy = {kD_xy:.4f}  kD_z = {kD_z:.4f}")

In [ ]:
model = PlantModel(
    mass_kg=float(MASS_KG), kT=float(kT), tau_motor_s=float(tau_motor),
    inertia=Inertia(Ixx=float(Ixx), Iyy=float(Iyy), Izz=float(Izz)),
    drag=Drag(kD_xy=float(kD_xy), kD_z=float(kD_z)),
    arm_length_m=float(arm),
    source_log=LOG_PATH,
    fit_rmse={},
)
model.save(ARTIFACT_PATH)
print(f"Saved to {ARTIFACT_PATH}")
import json
import dataclasses
def _json_default(o):
    if hasattr(o, 'item'): return o.item()
    return str(o)
print(json.dumps(dataclasses.asdict(model), default=_json_default, indent=2))

In [ ]:
m = PlantModel.load(ARTIFACT_PATH)

I_vec = np.array([m.inertia.Ixx, m.inertia.Iyy, m.inertia.Izz])
sim_omega = np.zeros(3)
sim_omegas = []
for i in range(len(t_ref) - 1):
    dt_i = t_ref[i+1] - t_ref[i]
    tau = np.array([tau_roll[i], tau_pitch[i], tau_yaw[i]])
    alpha = (tau - np.cross(sim_omega, I_vec * sim_omega)) / I_vec
    alpha *= dt_i / (m.tau_motor_s + dt_i)
    sim_omega = np.clip(sim_omega + alpha * dt_i, -100, 100)  # clamp runaway
    sim_omegas.append(sim_omega.copy())

sim_omegas = np.array(sim_omegas)
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
labels = ["Roll rate (rad/s)", "Pitch rate (rad/s)", "Yaw rate (rad/s)"]
for i, label in enumerate(labels):
    real_sig = [p_rate, q_rate, r_rate][i]
    axes[i].plot(t_ref[1:], sim_omegas[:, i], label="simulated", alpha=0.8)
    axes[i].plot(t_ref, real_sig, label="real", alpha=0.6)
    axes[i].set_ylabel(label); axes[i].legend()
axes[0].set_title("Model validation: simulated vs real angular rates")
axes[-1].set_xlabel("Time (s)")
plt.tight_layout(); plt.savefig("/tmp/sysid_validation.png"); plt.close()
print("Saved validation plot to /tmp/sysid_validation.png")

rmse_roll  = np.sqrt(np.mean((sim_omegas[:, 0] - p_rate[1:])**2))
rmse_pitch = np.sqrt(np.mean((sim_omegas[:, 1] - q_rate[1:])**2))
if np.isfinite(rmse_roll) and np.isfinite(rmse_pitch):
    print(f"RMSE — roll: {rmse_roll:.4f} rad/s, pitch: {rmse_pitch:.4f} rad/s")
else:
    print("Open-loop replay diverged (expected without feedback). Inertia estimates are approximate.")